# Model Governance, Stability (PSI / CSI) & Fair Lending Compliance

In institutional finance and tier-1 banking, building an accurate XGBoost model is only step one. Regulatory frameworks (SR 11-7, Basel III, ECOA/Equal Credit Opportunity Act) mandate continuous model monitoring, concept drift detection, and rigorous algorithmic fairness auditing.

In this notebook, we implement:
1. **Population Stability Index (PSI)**: Detecting distribution shifts in model scores over time.
2. **Characteristic Stability Index (CSI)**: Pinpointing which specific features drove the score drift.
3. **Fair Lending & Disparate Impact Auditing**: Enforcing the **Four-Fifths (80%) Rule** and measuring Equal Opportunity differences across protected demographic cohorts.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Population Stability Index (PSI)

PSI measures how much a variable or model score distribution has shifted between a **Baseline (Expected / Development)** population and a **Current (Actual / Production)** population.

### Mathematical Definition
For $B$ bins (typically deciles determined on the baseline population):

$$\text{PSI} = \sum_{b=1}^{B} \left( \% \text{Actual}_b - \% \text{Expected}_b \right) \times \ln\left( \frac{\% \text{Actual}_b + \epsilon}{\% \text{Expected}_b + \epsilon} \right)$$

Where $\epsilon = 10^{-6}$ prevents division by zero or $\ln(0)$.

### Standard Regulatory Decision Rules:
- $\text{PSI} < 0.10$: **🟢 Green (No Change)** - Model score distribution is stable.
- $0.10 \le \text{PSI} < 0.25$: **🟡 Amber (Moderate Shift)** - Warning: Review feature inputs, increase monitoring frequency.
- $\text{PSI} \ge 0.25$: **🔴 Red (Significant Shift)** - Critical Alert: Model degradation likely; mandatory retrain or governance sign-off.

In [ ]:
def calculate_psi(expected: np.ndarray, actual: np.ndarray, num_bins: int = 10, epsilon: float = 1e-6) -> tuple:
    """
    Computes Population Stability Index (PSI) using quantile binning defined on the expected dataset.
    Returns (total_psi, psi_breakdown_df).
    """
    # 1. Define bin thresholds based on expected (baseline) quantiles
    quantiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(expected, quantiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    # Deduplicate edges if distribution has ties
    bin_edges = np.unique(bin_edges)
    
    # 2. Compute counts in each bin
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # 3. Convert to percentages
    pct_expected = (expected_counts / len(expected)) + epsilon
    pct_actual = (actual_counts / len(actual)) + epsilon
    
    # Re-normalize after epsilon addition
    pct_expected /= pct_expected.sum()
    pct_actual /= pct_actual.sum()
    
    # 4. PSI calculation per bin
    psi_bins = (pct_actual - pct_expected) * np.log(pct_actual / pct_expected)
    total_psi = float(np.sum(psi_bins))
    
    # Format breakdown dataframe
    breakdown = pd.DataFrame({
        'Bin': [f"[{bin_edges[i]:.3f}, {bin_edges[i+1]:.3f})" for i in range(len(bin_edges)-1)],
        'Expected_Pct': pct_expected,
        'Actual_Pct': pct_actual,
        'PSI_Contribution': psi_bins
    })
    
    return total_psi, breakdown

## 2. Simulating Production Model Drift & CSI Attribution

Let's train a credit scoring XGBoost model on baseline data (Month 0), and then simulate macroeconomic stress in production (Month +6, where unemployment increases and debt-to-income spikes).

In [ ]:
# Generate Baseline Data (Month 0)
n_baseline = 10000
dti_base = np.random.beta(2, 5, n_baseline) * 0.6
credit_score_base = np.random.normal(700, 50, n_baseline)
income_base = np.random.lognormal(10.8, 0.4, n_baseline)

X_base = pd.DataFrame({'dti': dti_base, 'credit_score': credit_score_base, 'income': income_base})
log_odds = -2.5 + 4.5 * X_base['dti'] - 0.008 * (X_base['credit_score'] - 600) - 0.00001 * (X_base['income'] - 50000)
prob_default_base = 1 / (1 + np.exp(-log_odds))
y_base = (np.random.rand(n_baseline) < prob_default_base).astype(int)

# Train Baseline Model
model = xgb.XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.08, random_state=42)
model.fit(X_base, y_base)
scores_base = model.predict_proba(X_base)[:, 1]

# ---------------------------------------------------------
# Simulate Production Cohorts (Stable vs. Macro Shock)
# ---------------------------------------------------------
n_prod = 5000

# Scenario A: Stable Cohort (T+1 Month)
X_stable = pd.DataFrame({
    'dti': np.random.beta(2, 5, n_prod) * 0.6,
    'credit_score': np.random.normal(700, 50, n_prod),
    'income': np.random.lognormal(10.8, 0.4, n_prod)
})
scores_stable = model.predict_proba(X_stable)[:, 1]

# Scenario B: Macro Shock Cohort (T+6 Months: Spiking DTI and Lower Credit Scores)
X_shock = pd.DataFrame({
    'dti': np.random.beta(3.5, 4, n_prod) * 0.7, # Severe DTI increase
    'credit_score': np.random.normal(650, 60, n_prod), # Credit score degradation
    'income': np.random.lognormal(10.6, 0.5, n_prod) # Income decline
})
scores_shock = model.predict_proba(X_shock)[:, 1]

# Evaluate PSI
psi_stable, _ = calculate_psi(scores_base, scores_stable)
psi_shock, breakdown_shock = calculate_psi(scores_base, scores_shock)

print(f"PSI for T+1 Month (Stable Cohort):    {psi_stable:.4f} -> {'[STABLE]' if psi_stable < 0.1 else '[WARNING]'}")
print(f"PSI for T+6 Month (Macro Shock):      {psi_shock:.4f} -> {'[CRITICAL DRIFT]' if psi_shock >= 0.25 else '[WARNING]'}")

### Visualizing PSI Score Shift & Feature CSI

In [ ]:
# Characteristic Stability Index (CSI) for individual features
csi_summary = []
for feature in X_base.columns:
    feat_csi, _ = calculate_psi(X_base[feature].values, X_shock[feature].values)
    csi_summary.append({'Feature': feature, 'CSI': feat_csi})

df_csi = pd.DataFrame(csi_summary).sort_values(by='CSI', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Score Distribution Shift
axes[0].hist(scores_base, bins=40, alpha=0.5, color='navy', density=True, label='Baseline (Month 0)')
axes[0].hist(scores_shock, bins=40, alpha=0.5, color='crimson', density=True, label=f'Production Month 6 (PSI={psi_shock:.3f})')
axes[0].set_title("Model Output Score Distribution Drift")
axes[0].set_xlabel("Predicted Probability of Default")
axes[0].set_ylabel("Density")
axes[0].legend()

# Plot 2: CSI Feature Drift Attribution
colors = ['crimson' if x >= 0.25 else 'orange' if x >= 0.1 else 'seagreen' for x in df_csi['CSI']]
axes[1].barh(df_csi['Feature'], df_csi['CSI'], color=colors)
axes[1].axvline(0.1, color='orange', linestyle='--', label='Warning Threshold (0.10)')
axes[1].axvline(0.25, color='red', linestyle='--', label='Critical Threshold (0.25)')
axes[1].set_title("Characteristic Stability Index (CSI) Attribution")
axes[1].set_xlabel("CSI Metric")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Fair Lending Compliance: Adverse Impact & 80% Rule

Under fair lending laws (e.g., US Equal Credit Opportunity Act), models must not disproportionately disadvantage protected demographic classes (such as Age, Gender, Ethnicity/Region proxies).

### Key Fairness Metrics:
1. **Disparate Impact (Four-Fifths / 80% Rule)**:
   $$\text{Disparate Impact Ratio} = \frac{\text{Approval Rate}_{\text{Protected}}}{\text{Approval Rate}_{\text{Reference}}} \ge 0.80$$
   If the ratio is below 0.80, the model has presumptive adverse impact.

2. **Equal Opportunity Difference (True Positive Rate Parity)**:
   $$\Delta \text{TPR} = |\text{TPR}_{\text{Protected}} - \text{TPR}_{\text{Reference}}|$$
   Measures whether creditworthy applicants in both groups have equal probability of being approved.

In [ ]:
# Simulate dataset with Protected Demographic Group (e.g., Young Applicants < 25 vs Mature Applicants >= 25)
n_audit = 8000
is_young = (np.random.rand(n_audit) < 0.3).astype(int) # 30% protected minority group

# Young applicants have shorter credit history (lower credit score on average)
credit_scores_audit = np.where(is_young == 1, np.random.normal(660, 45, n_audit), np.random.normal(710, 45, n_audit))
dti_audit = np.random.beta(2, 5, n_audit) * 0.6
income_audit = np.random.lognormal(10.8, 0.4, n_audit)

X_audit = pd.DataFrame({'dti': dti_audit, 'credit_score': credit_scores_audit, 'income': income_audit})
# True underlying default probability
true_log_odds = -2.5 + 4.5 * dti_audit - 0.008 * (credit_scores_audit - 600)
y_audit = (np.random.rand(n_audit) < (1 / (1 + np.exp(-true_log_odds)))).astype(int)

# Model score predictions (Probability of Default)
pred_pd = model.predict_proba(X_audit)[:, 1]

# Standard Global Approval Threshold (e.g. Approve if Predicted PD < 0.20)
APPROVAL_THRESHOLD = 0.20
approved = (pred_pd < APPROVAL_THRESHOLD).astype(int)

# Compute Approval Rates
app_rate_ref = np.mean(approved[is_young == 0])
app_rate_prot = np.mean(approved[is_young == 1])
di_ratio = app_rate_prot / app_rate_ref

# Compute True Positive Rate for Creditworthy Individuals (y_audit == 0)
tpr_ref = np.mean(approved[(is_young == 0) & (y_audit == 0)])
tpr_prot = np.mean(approved[(is_young == 1) & (y_audit == 0)])
eod = abs(tpr_ref - tpr_prot)

print("=== FAIR LENDING COMPLIANCE AUDIT ===")
print(f"Reference Group (Mature) Approval Rate:   {app_rate_ref * 100:.2f}%")
print(f"Protected Group (Young)  Approval Rate:   {app_rate_prot * 100:.2f}%")
print(f"Disparate Impact Ratio (4/5ths Rule):    {di_ratio:.3f} -> {'[PASSED] PASSED (>= 0.80)' if di_ratio >= 0.8 else '[FAILED] FAILED (Adverse Impact)'}")
print(f"Equal Opportunity Difference (|Delta TPR|):   {eod * 100:.2f}%")

## 4. Fair Threshold Calibration (Mitigating Adverse Impact)

When a single threshold violates fair lending standards, we can tune group-specific decision thresholds to restore the Disparate Impact ratio above 0.80 while preserving overall portfolio risk limits.

In [ ]:
thresholds = np.linspace(0.10, 0.40, 100)
di_curve = []

for t_prot in thresholds:
    app_prot = np.mean(pred_pd[is_young == 1] < t_prot)
    di_curve.append(app_prot / app_rate_ref)

# Find threshold that achieves DI >= 0.85
fair_threshold_prot = thresholds[np.where(np.array(di_curve) >= 0.85)[0][0]]
remedied_approved = np.where(is_young == 1, pred_pd < fair_threshold_prot, pred_pd < APPROVAL_THRESHOLD)
new_di_ratio = np.mean(remedied_approved[is_young == 1]) / np.mean(remedied_approved[is_young == 0])

print(f"Adjusted Protected Threshold: {fair_threshold_prot:.3f} (Reference: {APPROVAL_THRESHOLD:.3f})")
print(f"Remediated Disparate Impact Ratio: {new_di_ratio:.3f} (Compliant!)")

# Plot Fair Threshold Tuning
plt.figure(figsize=(9, 4.5))
plt.plot(thresholds, di_curve, color='navy', linewidth=2.5, label='Disparate Impact vs Threshold')
plt.axhline(0.80, color='crimson', linestyle='--', label='80% Legal Threshold Barrier')
plt.axvline(fair_threshold_prot, color='seagreen', linestyle=':', label=f'Fair Threshold ({fair_threshold_prot:.3f})')
plt.title("Fair Lending Threshold Calibration for Protected Class")
plt.xlabel("Decision Cutoff for Protected Class")
plt.ylabel("Disparate Impact Ratio")
plt.legend()
plt.show()